<a href="https://colab.research.google.com/github/Raijeku/quantum-eigengame/blob/main/EigenGame_theory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [1]:
import autograd.numpy as np
import matplotlib.pyplot as plt
#import pennylane as qml
#from pennylane import qaoa
#from pennylane import numpy as np
from scipy.linalg import expm, sqrtm, eigh
import networkx as nx
from jax import grad
import jax.numpy as jnp
import numpy

# Functions

In [57]:
from jax import vjp

def generate_matrix(eigenvalues):
  D = np.diag(eigenvalues)
  P = np.random.rand(D.shape[0], D.shape[0])
  norm = np.linalg.norm(P)
  P = P/norm
  np.fill_diagonal(P, 1)
  return np.dot(P, np.dot(D, np.linalg.inv(P))), P

# Variational functions

def U3(theta_3):
  return jnp.array([[jnp.cos(theta_3[0]/2), -jnp.exp(1j*theta_3[2])*jnp.sin(theta_3[0]/2)],
                   [jnp.exp(1j*theta_3[1])*jnp.sin(theta_3[0]/2), jnp.exp(1j*(theta_3[1]+theta_3[2]))*jnp.cos(theta_3[0]/2)]])

def ansatz(theta):
  qubits, layers = theta.shape[:2]
  n = 2**qubits

  U = jnp.eye(n)
  for l in range(layers):
    U_l = U3(theta[0,l])
    for k in range(1, qubits):
      U_l = jnp.kron(U_l, U3(theta[k, l]))
    U = jnp.matmul(U, U_l)

  return U

def v(theta_index):
  qubits = theta_index.shape[0]
  n = 2**qubits

  s = jnp.ones(n)
  s /= jnp.linalg.norm(s)

  phi = jnp.matmul(ansatz(theta_index), s)

  return phi

def d_v(theta_index):
  return grad(v)(theta_index)

def rewards_var(theta_i, X):
  return jnp.linalg.norm(X @ v(theta_i))**2

def penalties_var(theta_i, X, thetas=[]):
  return jnp.array([(jnp.conj(X @ v(theta_i)) @ X @ v(theta_j))**2 / (jnp.linalg.norm(X @ v(theta_i)))**2 for theta_j in thetas]).sum(axis=0)

def agent_utility_var(theta_i, X, thetas=[]):
  return rewards_var(theta_i, X) - penalties_var(theta_i, X, thetas)

def d_rewards_var(theta_i, X):
  return grad(rewards_var)(theta_i, X)

def d_penalties_var(theta_i, X, thetas=[]):
  if len(thetas) == 0:
    return grad(penalties_var)(theta_i, X, thetas)
  else:
    return grad(penalties_var, holomorphic = True)(theta_i, X, thetas)

def d_agent_utility_var(theta_i, X, thetas=[]):
  return d_rewards_var(theta_i, X) - d_penalties_var(theta_i, X, thetas)

# Variational tests

qubits = 3
layers = 2
n = 2**qubits

# Using U3 gates
theta_init = np.random.rand(qubits, layers, 3)

eigs = np.random.rand(n)
X, _ = generate_matrix(eigs)

print(agent_utility_var(theta_init, X))

print(d_agent_utility_var(theta_init, X), d_agent_utility_var(theta_init, X).shape)

# Normal functions

def rewards(v_i, X):
  return jnp.linalg.norm(X @ v_i)**2

def penalties(v_i, X, v=[]):
  return jnp.array([(jnp.conj(X @ v_i) @ X @ v_j)**2 / (jnp.linalg.norm(X @ v_i))**2 for v_j in v]).sum(axis=0)

def agent_utility(v_i, X, v=[]):
  return rewards(v_i, X) - penalties(v_i, X, v)

def d_rewards(v_i, X):
  return grad(rewards)(v_i, X)

def d_penalties(v_i, X, v=[]):
  return grad(penalties)(v_i, X, v)
  #if len(v) == 0:
  #  return grad(penalties)(v_i, X, v)
  #else:
  #  return grad(penalties, holomorphic = True)(v_i, X, v)

def d_agent_utility(v_i, X, v=[]):
  return d_rewards(v_i, X) - d_penalties(v_i, X, v)

def d_agent_utility_R(v_i, X, v=[]):
  grad_u = d_rewards(v_i, X) - d_penalties(v_i, X, v)

  return grad_u - np.dot(grad_u.T, v_i) * v_i

# Normal tests
n = 2**qubits

# Using U3 gates
v_init = np.random.rand(n)

eigs = np.random.rand(n)
X, _ = generate_matrix(eigs)

print(agent_utility(v_init, X))

print(d_agent_utility(v_init, X), d_agent_utility(v_init, X).shape)

0.6289214
[[[ 0.02983777  0.00089049 -0.00724404]
  [ 0.03351778 -0.00724408 -0.01394065]]

 [[-0.00547347  0.00708743  0.00260763]
  [ 0.00989647  0.00260765 -0.00312249]]

 [[ 0.20381287 -0.0055275   0.10849768]
  [-0.1811983   0.10849764  0.17544219]]] (3, 2, 3)
0.9618689
[ 0.96931267  0.38410515  0.41175655  1.0414042   0.8418244   0.05173419
  0.3919337  -0.5120739 ] (8,)


In [74]:
def randomize_angle(ref_angle):
  sat_angle = np.random.uniform(low=0, high=ref_angle)

  return sat_angle

def initialize_relative_vec(ref_angle, ref_vec):
  a = np.random.rand(n)
  a -= a.dot(eigvecs[0]) * ref_vec
  a /= np.linalg.norm(a)

  v = eigvecs[0]*np.cos(ref_angle) + a*np.sin(ref_angle)

  return v

In [59]:
eigs = np.random.rand(n)
X, _ = generate_matrix(eigs)
M = X.T @ X
M /= np.linalg.norm(M)

eigs, eigvecs = np.linalg.eig(M)
eigs, eigvecs = zip(*sorted(zip(eigs, eigvecs), reverse=True))
print('Eigenvalues:', eigs)
print('Eigenvectors:', eigvecs)

Eigenvalues: (0.7277663992451883, 0.5012956771707182, 0.39943060723849966, 0.21276653115894936, 0.1155888868822136, 0.02799855721489605, 0.009976035836054461, 0.00027227105628454114)
Eigenvectors: (array([ 0.01330157, -0.03766779, -0.08982603, -0.09741524,  0.02982252,
        0.05280197, -0.06318463,  0.98649685]), array([-0.92768713, -0.31504239, -0.10990537, -0.06637541, -0.05380757,
       -0.14286346, -0.01709657, -0.00790443]), array([-0.0850235 ,  0.03546438,  0.05485795,  0.97908848, -0.06389212,
        0.01071393, -0.12731541,  0.09738292]), array([ 0.05739128, -0.04153949, -0.1401004 , -0.10925376, -0.08178132,
        0.0296388 , -0.97376218, -0.08738863]), array([-0.13164679,  0.04646948,  0.97084082, -0.09264937,  0.02017232,
        0.05447557, -0.14534142,  0.06996624]), array([-0.07167532,  0.07850027, -0.03739967,  0.04902389,  0.98609469,
       -0.06083459, -0.08994577, -0.02691575]), array([ 0.16659699, -0.06795275,  0.0741777 ,  0.01135142, -0.04184147,
       -0.

# Lemma O.6 from original EigenGame paper

In [81]:
print('Testing Lemma O.6 from original EigenGame paper')

for attempt in range(10):
  print()
  print(f'Attempt {attempt}:')

  g = [- (eigs[i+1] - eigs[i]) for i in range(len(eigs)-1)]
  print('g:', g)

  c_i = 1/16
  print('c_i:', c_i)

  # For i = 1
  i = 0
  angle_dif = np.pi/4
  rand_angle = randomize_angle(angle_dif)
  v_i = initialize_relative_vec(rand_angle, eigvecs[i])

  lhs = np.abs(rand_angle)
  rhs = np.linalg.norm(d_agent_utility(v_i, M)) * np.pi/g[i]

  print(f'i = 1, {lhs} <= {rhs}')

  # For i = 2
  i = 1
  angle_dif = np.pi/4
  rand_angle = randomize_angle(angle_dif)
  v_i = initialize_relative_vec(rand_angle, eigvecs[i])

  vecs = [initialize_relative_vec(randomize_angle(c_i*g[i]/(i*eigs[0])), eigvecs[j]) for j in range(i)]

  lhs = np.abs(rand_angle)
  rhs = np.linalg.norm(d_agent_utility(v_i, M, vecs)) * np.pi/g[i]

  print(f'i = 2, {lhs} <= {rhs}')

  # For i = 3
  i = 2
  angle_dif = np.pi/4
  rand_angle = randomize_angle(angle_dif)
  v_i = initialize_relative_vec(rand_angle, eigvecs[i])

  vecs = [initialize_relative_vec(randomize_angle(c_i*g[i]/(i*eigs[0])), eigvecs[j]) for j in range(i)]

  lhs = np.abs(rand_angle)
  rhs = np.linalg.norm(d_agent_utility(v_i, M, vecs)) * np.pi/g[i]

  print(f'i = 3, {lhs} <= {rhs}')

  # For i = 4
  i = 3
  angle_dif = np.pi/4
  v_i = initialize_relative_vec(angle_dif, eigvecs[i])

  vecs = [initialize_relative_vec(randomize_angle(c_i*g[i]/(i*eigs[0])), eigvecs[j]) for j in range(i)]

  lhs = np.abs(angle_dif)
  rhs = np.linalg.norm(d_agent_utility(v_i, M, vecs)) * np.pi/g[i]

  print(f'i = 4, {lhs} <= {rhs}')

  # For i = 5
  i = 4
  angle_dif = np.pi/4
  rand_angle = randomize_angle(angle_dif)
  v_i = initialize_relative_vec(rand_angle, eigvecs[i])

  vecs = [initialize_relative_vec(randomize_angle(c_i*g[i]/(i*eigs[0])), eigvecs[j]) for j in range(i)]

  lhs = np.abs(rand_angle)
  rhs = np.linalg.norm(d_agent_utility(v_i, M, vecs)) * np.pi/g[i]

  print(f'i = 5, {lhs} <= {rhs}')

  # For i = 6
  i = 5
  angle_dif = np.pi/4
  rand_angle = randomize_angle(angle_dif)
  v_i = initialize_relative_vec(rand_angle, eigvecs[i])

  vecs = [initialize_relative_vec(randomize_angle(c_i*g[i]/(i*eigs[0])), eigvecs[j]) for j in range(i)]

  lhs = np.abs(rand_angle)
  rhs = np.linalg.norm(d_agent_utility(v_i, M, vecs)) * np.pi/g[i]

  print(f'i = 6, {lhs} <= {rhs}')

  # For i = 7
  i = 6
  angle_dif = np.pi/4
  rand_angle = randomize_angle(angle_dif)
  v_i = initialize_relative_vec(rand_angle, eigvecs[i])

  vecs = [initialize_relative_vec(randomize_angle(c_i*g[i]/(i*eigs[0])), eigvecs[j]) for j in range(i)]

  lhs = np.abs(rand_angle)
  rhs = np.linalg.norm(d_agent_utility(v_i, M, vecs)) * np.pi/g[i]

  print(f'i = 7, {lhs} <= {rhs}')



Testing Lemma O.6 from original EigenGame paper

Attempt 0:
g: [0.22647072207447005, 0.10186506993221855, 0.1866640760795503, 0.09717764427673577, 0.08759032966731754, 0.01802252137884159, 0.00970376477976992]
c_i: 0.0625
i = 1, 0.38390884411281806 <= 1.4959376501156583
i = 2, 0.2865774187469841 <= 5.006081808655884
i = 3, 0.454622128825284 <= 6.423936316646618
i = 4, 0.7853981633974483 <= 8.031719603964625
i = 5, 0.7593356583842541 <= 12.592827323376444
i = 6, 0.5441952598931644 <= 37.115433711260685
i = 7, 0.1561574281970879 <= 56.53050757315266

Attempt 1:
g: [0.22647072207447005, 0.10186506993221855, 0.1866640760795503, 0.09717764427673577, 0.08759032966731754, 0.01802252137884159, 0.00970376477976992]
c_i: 0.0625
i = 1, 0.4748557362241339 <= 0.8622092690848253
i = 2, 0.05382149106125794 <= 6.201098576993032
i = 3, 0.22751413713096091 <= 3.088188774617609
i = 4, 0.7853981633974483 <= 5.290892671511759
i = 5, 0.7057676315728959 <= 22.243878592397095
i = 6, 0.60808484474166 <= 123.53